In [70]:
import torch.nn.functional as F
from torch import nn
import torch

# 读取数据
DATA_PATH = "/Users/wangdajin/我的文稿/pycharm/github项目学习/names.txt"
with open(DATA_PATH, "r", encoding="utf-8") as f:
    words = [line.strip() for line in f if line.strip()]
abcd=['.']+sorted(set(''.join(words)))
print(abcd)

# 构造训练样本
ch_i = {ch: i for i,ch in enumerate(abcd)}
i_ch={i:ch for ch,i in ch_i.items()}
print(ch_i.items())
a1 = []
a2=[]
for word in words:
    word_=["."] + list(word) + ["."]
    for ch1,ch2 in zip(word_,word_[1:]):
        a1.append(ch_i[ch1])
        a2.append(ch_i[ch2])
idx = torch.tensor(a1, dtype=torch.long).view(-1, 1)
targets = torch.tensor(a2, dtype=torch.long).view(-1, 1)
assert idx.shape == targets.shape, "每个输入必须有一个对应答案"
# 定义模型

class bigram(nn.Module):
    def __init__(self, a=len(abcd) ):
        super().__init__()#将父类的self中定义的变量加入到这个class中 在这个模型中就是把nnmodule中的变量加到这里
        self.logits = nn.Embedding (len(abcd), len(abcd))
        self.loss = nn.CrossEntropyLoss()

# 前向传播
    def forward(self, idx , targets=None): #避免targets为0 如果有值none不会覆盖
        logits = self.logits.weight[idx]
        loss = None
        if targets is not None:
            B,T,C = logits.size()
            logits = logits.view(B*T,C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits,targets)
        return logits,loss
# 开始训练
model = bigram(len(abcd))
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)

for step in range(1):
    logits,loss = model(idx,targets)
    optimizer.zero_grad()#清掉上次的梯度
    loss.backward()
    optimizer.step()
    if step % 100 == 0:
        print(step,loss.item())

# 开始生成
@torch.no_grad()
def generate():
    current = torch.tensor([[ch_i["."]]], dtype=torch.long)
    name = []
    k=0
    while True:

        logits,_ = model(current)
        next_logits = logits[0, -1]

        probs = F.softmax(next_logits, dim=0)
        next_id = torch.multinomial(probs, num_samples=1).item()#按概率生成
        if k == 5 or next_id == ch_i['.']:

            break

        name.append(i_ch[next_id])
        current = torch.tensor([[next_id]], dtype=torch.long)
        k+=1
    return ''.join(name)

for i in range(10):
    print(generate())




['.', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
dict_items([('.', 0), ('a', 1), ('b', 2), ('c', 3), ('d', 4), ('e', 5), ('f', 6), ('g', 7), ('h', 8), ('i', 9), ('j', 10), ('k', 11), ('l', 12), ('m', 13), ('n', 14), ('o', 15), ('p', 16), ('q', 17), ('r', 18), ('s', 19), ('t', 20), ('u', 21), ('v', 22), ('w', 23), ('x', 24), ('y', 25), ('z', 26)])
0 3.7194983959198
jiqcs
yoktn
jprzu
gu
yquqp
zzimb
psyra
akgfn
jxixm
quwgi
